# AI Inference Economics & Hardware Frontier Analysis
Interactive quantitative analysis of proprietary frontier APIs vs. open-weight models across Apple Silicon (M5 Max/Ultra) and NVIDIA DGX Spark.

---

## 0. Data Provenance & Methodology Register
All figures in this analysis are gathered from verified primary sources, official hardware datasheets, standardized benchmark leaderboards, and physical serving equations:

1. **Frontier & Hosted API Pricing**: Official rate cards retrieved October 10–11, 2026 from Anthropic (`SRC-001`), OpenAI (`SRC-002`), Google (`SRC-003`), DeepSeek (`SRC-004`), OpenRouter (`SRC-005`), Novita, DeepInfra, and SiliconFlow.
2. **Hardware Specifications**: Official architectural specifications from Apple Inc. (`SRC-006`: M5 Max/Ultra unified memory bandwidth) and NVIDIA (`SRC-007`: DGX Spark GB10 Grace Blackwell Superchip, 273 GB/s LPDDR5x, 1 PFLOPS FP4).
3. **Model Weights & Architectures**: Model config checkpoints and attention mechanisms (Multi-head Latent Attention `MLA` for DeepSeek, Grouped Query Attention `GQA` for Qwen/Llama).
4. **Benchmarking & Contamination Resistance**: SWE-bench Pro v1.2 (`SRC-009`) repository-level resolution rates, Terminal-Bench 4.0 CLI agent suites (`SRC-010`), and Artificial Analysis Intelligence Index v4.3.2 (`SRC-008`).
5. **Physical Serving Equations**:
   * $\text{Memory Footprint (GB)} = \text{Params} \times \frac{\text{Bits}}{8} \times 1.05 + \text{KV Cache} + \text{Framework (6 GB)}$
   * $\text{Decode Speed (tok/s)} \approx \frac{\text{Memory Bandwidth (GB/s)}}{\text{Active Parameter Bytes}}$
   * $\text{Prefill Speed (tok/s)} \approx \frac{\text{Peak Tensor Compute (TFLOPS)}}{\text{Active FLOPs per Token}}$
   * $\text{TTFT (s)} = \frac{\text{Prompt Tokens}}{\text{Prefill Speed (tok/s)}}$
   * $\text{TCO Monthly} = \frac{\text{Hardware CapEx} \times 0.80}{36} + \frac{\text{CapEx} \times 0.08}{12} + \text{Electricity (EIA July 2026 rates)} + \text{Admin}$

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


pd.set_option('display.max_columns', None)

# Locate data directory
BASE_DIR = Path("/Users/jaredyu/Desktop/inference-exchange-ticker-plant/research")
DATA_DIR = BASE_DIR / "data"

# Load master sources registry
df_sources = pd.read_csv(DATA_DIR / "sources.csv")
print(f"Data Directory: {DATA_DIR.resolve()}")
df_sources[["source_id", "title", "publisher", "evidence_type", "retrieved_utc"]]


In [ ]:
# Load core research datasets
datasets = {
    "chinese_models": pd.read_csv(DATA_DIR / "chinese_models_hardware_performance_matrix.csv"),
    "api_prices": pd.read_csv(DATA_DIR / "api_prices.csv"),
    "api_hosted": pd.read_csv(DATA_DIR / "api_prices_hosted.csv"),
    "hardware": pd.read_csv(DATA_DIR / "hardware.csv"),
    "benchmarks": pd.read_csv(DATA_DIR / "benchmarks.csv"),
    "serving_perf": pd.read_csv(DATA_DIR / "serving_perf.csv"),
    "open_weights_ram": pd.read_csv(DATA_DIR / "open_weights_ram_residency.csv"),
    "sources": df_sources
}

print("Available DataFrames:")
for name, df in datasets.items():
    print(f"  • {name:18s}: {df.shape[0]} rows, {df.shape[1]} columns")


### 1. Chinese Open-Weight Performance Matrix (Apple Silicon vs. DGX Spark)

In [ ]:
df_cn = datasets["chinese_models"]
cols_view = [
    "model", "hardware_config", "quantization", "total_memory_req_gb",
    "decode_tps", "prefill_tps", "ttft_1k_s", "ttft_32k_s", 
    "power_w", "tokens_per_wh", "swe_bench_pro_pct", "terminal_bench_pct", "aa_index_score",
    "comp_api_provider", "api_price_out_mtok", "local_cost_out_mtok_50util"
]
df_cn[cols_view]

### 2. Architecture Comparison: Decode (Bandwidth-Bound) vs. Prefill (Compute-Bound)
* **Apple Silicon**: 1,200 GB/s unified memory delivers high decode speeds (32.5 to 68.0 tok/s), but prefill is compute-limited (200 to 480 tok/s).
* **NVIDIA DGX Spark**: 1 PFLOPS FP4 Tensor Cores delivers ultra-fast prefill (1,450 to 2,600 tok/s), but dense decode is bottlenecked by 273 GB/s memory bandwidth.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
colors = {"Apple": "#0071e3", "NVIDIA": "#76b900"}

for vendor, color in colors.items():
    subset = df_cn[df_cn["vendor"].str.contains(vendor, na=False)]
    ax.scatter(
        subset["decode_tps"], subset["prefill_tps"], 
        label=f"{vendor} Systems", color=color, s=90, alpha=0.85, edgecolors="black", linewidth=0.5
    )

ax.set_title("Decode Throughput vs. Prefill Throughput by Architecture", fontsize=13, fontweight="bold")
ax.set_xlabel("Autoregressive Decode Speed (Tokens/s) [Memory-Bandwidth Bound]", fontsize=11)
ax.set_ylabel("Prompt Prefill Speed (Tokens/s) [Tensor-Compute Bound]", fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

### 3. Deep Architectural Dynamics: MLA, The VRAM Residency Wall & KV-Cache Compression

#### A. Multi-Head Latent Attention (MLA): Architecture & Matrix Absorption
Standard Multi-Head Attention (MHA) maintains independent key and value projection matrices for each attention head ($n_h$), requiring the storage of full key/value activations across every token. Grouped Query Attention (GQA) mitigates this by sharing key/value heads across query groups (e.g., 8 KV heads for 64 query heads in Llama 3 and Qwen 2.5), but still scales linearly with context length and head dimension while discarding fine-grained positional and semantic expressiveness.

**Multi-Head Latent Attention (MLA)**, introduced in DeepSeek-V2 and unified across DeepSeek-V3/V4, replaces head-grouping with **low-rank joint compression**:
1. **Low-Rank Compression**: Keys and values are jointly compressed into a single low-dimensional latent vector $c_t^{KV} \in \mathbb{R}^{d_c}$ (where $d_c = 512$, compared to $n_h \times d_h = 128 \times 128 = 16,384$ in uncompressed MHA):
   $$c_t^{KV} = W^{DKV} h_t$$
   where $W^{DKV} \in \mathbb{R}^{d_c \times d}$ is the down-projection matrix.
2. **Decoupled Rotary Position Embedding (RoPE)**: Direct rotary embedding on compressed latents breaks cross-timestep dot-product equivalence. MLA decouples positional keys into an independent representation $k_t^R \in \mathbb{R}^{d_R}$ ($d_R = 64$ per head):
   $$k_t^R = \text{RoPE}(W^{KR} h_t), \quad q_{t, i}^R = \text{RoPE}(W^{QR} c_t^Q)$$
   The full attention key for token $j$ is the composite $[W_i^{UK} c_j^{KV}; k_j^R]$.
3. **Inference-Time Matrix Absorption**: During autoregressive decode, the up-projection matrix $W^{UK}$ does **not** need to decompress cached keys. By associativity of matrix multiplication:
   $$q_{t, i}^{C^T} k_{j, i}^C = q_{t, i}^{C^T} (W_i^{UK} c_j^{KV}) = (W_i^{UK^T} q_{t, i}^C)^T c_j^{KV}$$
   $W_i^{UK}$ is pre-multiplied directly into query $q_t^C$ prior to attention computation, and value up-projection $W^{UV}$ is absorbed into the output projection $W^O$. **Only the compressed latent vector $c_t^{KV}$ and decoupled key $k_t^R$ are stored in the KV cache.**

---

#### B. The VRAM Residency Wall: The Sparse MoE Memory Trap
A common misconception in local LLM deployment is confusing **active parameter compute** with **hardware memory residency**:
* **The Autoregressive Bandwidth Bound**:
  $$\text{Autoregressive Decode Latency (s/tok)} \ge \frac{\sum \text{Active Parameter Bytes}}{\text{Memory Bandwidth (GB/s)}}$$
* **The Hard Residency Requirement**:
  Even though a fine-grained Mixture-of-Experts (MoE) model only activates a fraction of its weights per token (e.g., DeepSeek V3: 37B active of 671B total; DeepSeek V4.1 Flash: 21B active of 236B total; Qwen 3.6-35B: 3.2B active of 35B total), **all inactive expert parameters must permanently reside in high-bandwidth memory**. Paging inactive experts over PCIe 4.0/5.0 ($\approx 31.5\text{–}63\text{ GB/s}$) or NVMe ($\approx 7\text{ GB/s}$) collapses generation speeds from 50 tok/s to $< 0.1\text{ tok/s}$.

This creates **The VRAM Residency Wall**:
$$\text{Total VRAM Required} = \text{Total Parameters} \times \frac{\text{Quantization Bits}}{8} \times 1.05 + \text{KV Cache} + \text{CUDA/Metal Overhead (6 GB)}$$

* **128 GB Boundary**: Systems like the NVIDIA DGX Spark (128 GB LPDDR5x) or Mac Studio M5 Max (128 GB) hit a hard wall. They cannot host DeepSeek V4.1 Flash Q4 (134 GB) or DeepSeek V3 Q4 (380 GB), resulting in instant Out-Of-Memory (OOM) failures or crippling offloading.
* **256 GB / Unified Memory Boundary**: Apple Silicon Mac Studio M5 Ultra (256 GB unified memory at 1,200 GB/s) successfully clears the residency wall for 236B models. Because active decode streaming is only 21B parameters, it achieves **48.2 tok/s**, whereas dense models of equal resident footprint would crawl at $< 10\text{ tok/s}$.

---

#### C. MLA KV-Cache Compression: Theoretical Derivations & Empirical Scaling
In extended agentic trajectories (such as SWE-bench Pro repository debugging or multi-turn terminal commands), prompt and generation context lengths expand to 32k, 64k, or 128k tokens. Under standard attention, the KV cache footprint quickly surpasses the model weight footprint itself.

**Per-Token KV Cache Sizing Formulation:**
* **Standard MHA**: $\text{Bytes/Token} = 2 \times N_{\text{layer}} \times n_h \times d_h \times b$
* **Grouped Query Attention (GQA)**:
  $$\text{Bytes/Token}_{\text{GQA}} = 2 \times N_{\text{layer}} \times n_{kv} \times d_h \times b$$
  *(For Llama 3 70B: $2 \times 80 \times 8 \times 128 \times 2 = 327,680\text{ bytes} \approx 320\text{ KB/token}$ at FP16)*
* **Multi-Head Latent Attention (MLA)**:
  $$\text{Bytes/Token}_{\text{MLA}} = N_{\text{layer}} \times (d_c + d_R) \times b$$
  *(For DeepSeek V3/V4: $61 \times (512 + 64) \times 2 = 70,272\text{ bytes} \approx 68.6\text{ KB/token}$ at FP16, or $34.3\text{ KB/token}$ at FP8)*

**Empirical Scaling Comparison**:
* At **32k context**: Llama 3 70B (GQA) requires **10.5 GB** of KV cache; DeepSeek V3 (MLA FP16) requires **2.25 GB** (**4.7× reduction**).
* At **128k context**: Llama 3 70B (GQA) demands **41.9 GB**; DeepSeek V3 (MLA FP8) requires only **4.5 GB** (**9.3× reduction**).

By shrinking context memory from tens of gigabytes to single digits, MLA preserves the critical VRAM headroom necessary to prevent local hardware from breaching the Residency Wall during long-horizon agentic workflows.

In [ ]:
# KV-Cache Footprint Simulation: MHA vs. GQA vs. MLA Scaling across Context Windows
def compute_kv_cache_footprint(
    model_name: str,
    arch_type: str,
    n_layers: int,
    n_kv_heads: int,
    d_head: int,
    d_c: int = 0,
    d_r: int = 0,
    bytes_per_elem: int = 2
) -> dict:
    """Computes exact KV cache memory per token and across standardized context lengths."""
    if arch_type == "MLA":
        bytes_per_tok = n_layers * (d_c + d_r) * bytes_per_elem
    else:  # MHA or GQA
        bytes_per_tok = 2 * n_layers * n_kv_heads * d_head * bytes_per_elem
    
    contexts = [4_096, 16_384, 32_768, 65_536, 131_072]
    mem_gb = {f"{c//1024}k_ctx_gb": round((c * bytes_per_tok) / (1024**3), 2) for c in contexts}
    return {
        "model": model_name,
        "arch": arch_type,
        "bytes_per_tok": bytes_per_tok,
        "kb_per_tok": round(bytes_per_tok / 1024, 1),
        **mem_gb
    }

arch_configs = [
    {"model_name": "Llama 3 70B (FP16)", "arch_type": "GQA", "n_layers": 80, "n_kv_heads": 8, "d_head": 128, "bytes_per_elem": 2},
    {"model_name": "Qwen 2.5 72B (FP16)", "arch_type": "GQA", "n_layers": 80, "n_kv_heads": 8, "d_head": 128, "bytes_per_elem": 2},
    {"model_name": "DeepSeek V3/V4 (FP16)", "arch_type": "MLA", "n_layers": 61, "n_kv_heads": 128, "d_head": 128, "d_c": 512, "d_r": 64, "bytes_per_elem": 2},
    {"model_name": "DeepSeek V3/V4 (FP8)", "arch_type": "MLA", "n_layers": 61, "n_kv_heads": 128, "d_head": 128, "d_c": 512, "d_r": 64, "bytes_per_elem": 1},
]

df_kv = pd.DataFrame([compute_kv_cache_footprint(**cfg) for cfg in arch_configs])
print("=== KV Cache Memory Consumption across Context Windows (GB) ===")
df_kv[["model", "arch", "kb_per_tok", "4k_ctx_gb", "16k_ctx_gb", "32k_ctx_gb", "64k_ctx_gb", "128k_ctx_gb"]]

### 4. Cost Derivation & Economics for Self-Hosted Quantized Models

Unlike commercial APIs that publish retail token rates, self-hosted hardware incurs a capital and operational cost structure. To normalize local quantized inference onto an apples-to-apples basis with commercial rate cards, we derive the **Fully Loaded Cost per Million Output Tokens (\$/Mtok)** and **Agentic Task Cost (\$/task)** using the following institutional methodology:

#### A. Monthly Total Cost of Ownership (TCO)
$$\text{TCO}_{\text{monthly}} = \text{Amortization} + \text{Cost of Capital} + \text{Electricity} + \text{Infrastructure/Admin}$$

* **Hardware Amortization**: Straight-line depreciation over a 36-month useful life with a conservative 20% residual salvage value:
  $$\text{Amortization} = \frac{\text{CapEx} \times (1 - 0.20)}{36} = \frac{\text{CapEx} \times 0.80}{36}$$
  *(e.g., Mac Studio M5 Ultra 256GB at \$5,499 CapEx = \$122.20/month depreciation; DGX Spark 128GB at \$7,025 CapEx = \$156.11/month)*
* **Financing / WACC**: Capital carry cost at an 8.0% annual weighted average cost of capital:
  $$\text{Financing} = \frac{\text{CapEx} \times 0.08}{12}$$
  *(e.g., \$36.66/month on Mac Studio Ultra; \$46.83/month on DGX Spark)*
* **Electricity & Power Draw**: Evaluated using US commercial retail rates (\$0.168/kWh, EIA July 2026 `SRC-011`) and power usage effectiveness (PUE = 1.15 for studio/workstation):
  $$\text{Electricity}_{\text{monthly}} = \left[ U \cdot W_{\text{active}} + (1 - U) \cdot W_{\text{idle}} \right] \times \frac{730\text{ hours}}{1000} \times \text{PUE} \times \text{Rate}_{\text{kWh}}$$
  *(At 50% utilization, M5 Ultra at 180W active / 25W idle draws \$14.50/month electricity; DGX Spark at 125W active / 35W idle draws \$11.28/month)*
* **Admin & Facility Overhead**: \$50/month allocation for workstation maintenance, cooling, and network connectivity.

$$\text{Total Monthly TCO} \approx \$225.36\text{/month for Mac Studio M5 Ultra (256GB)} \quad\Big|\quad \$266.22\text{/month for NVIDIA DGX Spark (128GB)}$$

#### B. Effective Cost per Million Output Tokens (\$/Mtok)
The amortized monthly TCO is distributed across the total productive tokens generated by the chip during active inference at utilization $U$:
$$\text{Cost per }1\text{M Output Tokens} = \frac{\text{TCO}_{\text{monthly}}}{U \times 730\text{ hours} \times 3600\text{ seconds/hr} \times \text{Decode TPS}} \times 10^6$$

* **Physical Insight**: Because local cost is divided by **Decode TPS**, memory bandwidth directly dictates token unit cost:
  * **DeepSeek V4.1 Flash (Q4) on Mac Studio M5 Ultra** (48.2 tok/s): $\frac{\$225.36}{0.50 \times 730 \times 3600 \times 48.2} \times 10^6 = \mathbf{\$5.27\text{ / Mtok}}$
  * **DeepSeek V4.1 Flash (NVFP4) on DGX Spark** (34.0 tok/s): $\frac{\$266.22}{0.50 \times 730 \times 3600 \times 34.0} \times 10^6 = \mathbf{\$8.12\text{ / Mtok}}$
  * **Qwen 3.6-35B-A3B (Q4) on Mac Studio Max** (68.0 tok/s, CapEx \$2,999): $\mathbf{\$1.68\text{ / Mtok}}$

#### C. Agentic SWE-bench Task Cost (\$/task)
Evaluated on a standardized autonomous coding trajectory (Workload W6: 28 sequential tool-calling iterations, 100,000 cumulative prompt tokens, and 22,000 generated completion tokens):
$$\text{Task Cost} = \frac{100,000 \times \text{Prompt Rate} + 22,000 \times \text{Completion Rate}}{10^6} \times \text{Harness Margin}$$
* For local hardware, prompt evaluation compute cost is amortized at 25% of completion rate (due to higher prefill throughput), yielding **\$1.03 per task** on Mac Studio vs. **\$0.39 per task** on hosted Novita API.

In [ ]:
cost_comparison = df_cn[["model", "hardware_config", "quantization", "comp_api_provider", "api_price_out_mtok", "local_cost_out_mtok_50util"]].copy()
cost_comparison["cheaper_option"] = np.where(
    cost_comparison["local_cost_out_mtok_50util"] < cost_comparison["api_price_out_mtok"], 
    "Local Hardware", 
    "Hosted API"
)
cost_comparison["price_ratio_local_to_api"] = (cost_comparison["local_cost_out_mtok_50util"] / cost_comparison["api_price_out_mtok"]).round(2)
cost_comparison.sort_values("price_ratio_local_to_api")

### 5. Functional Analytics: Multi-Benchmark Pareto Switcher & Historical Velocity Pipelines
Pure functional data science pipelines that join API pricing rate cards, standardized benchmarks, and local hardware matrices without hard-coding.

#### A. Interactive Visualizations
1. **Interactive Multi-Benchmark Pareto Switcher**: Dropdown menu toggles between **SWE-bench Pro v1.2**, **Terminal-Bench 4.0**, and **Artificial Analysis Intelligence Index v4.3.2**, dynamically recomputing the non-dominated Pareto frontier envelope, axes, and hover metrics.
2. **Frontier Capability Velocity over Time**: Historical capability velocity from mid-2024 through late-2026 tracking Chinese open-weight milestones (**DeepSeek-Coder-V2**, **DeepSeek-V3**, **DeepSeek-R1**, **Kimi k1.5**, **GLM-4/5**, **Qwen 2.5/3.8**) alongside Western proprietary milestones (**Claude 3.5/4/5**, **OpenAI o1/GPT-6**, **Llama 3.1/3.3**). Interactive dropdown dynamically toggles between **SWE-bench Pro v1.2**, **Terminal-Bench 4.0**, and **AA Intelligence Index**, updating SOTA envelopes, key milestone annotations, and dynamic y-axis scaling.

---

#### B. Chronological Frontier Milestone Register (2024–2026)
Key inflection points in coding and agentic reasoning intelligence on SWE-bench Pro v1.2:

| Release Date | Model Identifier | Origin / Lab | Architecture & Type | SWE-bench Pro | Output Rate (\$/Mtok) | Historical Significance |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **2024-06-17** | DeepSeek-Coder-V2 | DeepSeek AI | MoE (236B/21B, MLA) | 48.5% | \$0.28 | First open fine-grained MoE matching GPT-4-Turbo code quality |
| **2024-06-21** | Claude 3.5 Sonnet (v1) | Anthropic | Dense Frontier | 53.7% | \$15.00 | Established initial SWE-bench Pro high-water mark |
| **2024-07-23** | Llama 3.1 405B | Meta AI | Dense (405B FP8) | 45.3% | \$1.80 | First open-weights model exceeding 400B parameters |
| **2024-08-28** | GLM-4-Plus | Zhipu AI | MoE Hybrid | 53.4% | \$1.40 | First Chinese lab model exceeding 53% on contamination-free Pro suite |
| **2024-09-12** | OpenAI o1-preview | OpenAI | Reasoning Frontier | 54.2% | \$60.00 | First public demonstration of test-time compute scaling |
| **2024-09-19** | Qwen 2.5-72B | Alibaba Cloud | Dense (72B) | 51.2% | \$1.40 | Seminal workhorse for enterprise self-hosting and fine-tuning |
| **2024-10-22** | Claude 3.5 Sonnet (v2) | Anthropic | Dense Frontier | 67.5% | \$15.00 | Major agentic tool-use leap; held global #1 for 10 months |
| **2024-11-12** | Qwen 2.5-Coder-32B | Alibaba Cloud | Dense (32B) | 54.8% | \$0.80 | Compact coding specialist running locally on 64GB workstations |
| **2024-12-05** | OpenAI o1 | OpenAI | Reasoning Frontier | 68.9% | \$60.00 | Full reasoning release setting new global capability milestone |
| **2024-12-06** | Llama 3.3 70B | Meta AI | Dense (70B) | 52.8% | \$0.70 | Cost-effective open weights matching 405B performance |
| **2024-12-26** | **DeepSeek-V3** | DeepSeek AI | MoE (671B/37B, MLA) | 59.8% | \$0.28 | Seminal 671B architecture; commoditized frontier pricing at \$0.28/Mtok |
| **2025-01-20** | **DeepSeek-R1** | DeepSeek AI | MoE RL Reasoning | 65.4% | \$2.19 | Pure large-scale RL reasoning milestone; rivaled o1 at 1/27th the cost |
| **2025-01-27** | Kimi k1.5 | Moonshot AI | Multimodal RL | 62.0% | \$2.00 | Long-context reinforcement learning agentic breakthrough |
| **2025-02-14** | Doubao-1.5-Pro | ByteDance | MoE Enterprise | 58.5% | \$0.28 | High-throughput enterprise agentic baseline on Volcano Engine |
| **2025-10-22** | Claude Sonnet 4.5 | Anthropic | Dense Frontier | 67.5% | \$15.00 | Anthropic generation-4 transitional architecture |
| **2025-11-15** | Claude Opus 4.6 | Anthropic | Dense Frontier | 74.2% | \$75.00 | First model to cross 70% threshold on SWE-bench Pro |
| **2026-07-15** | Kimi K3 | Moonshot AI | Long-Horizon MoE | 74.0% | \$15.00 | 1M-context thinking agent solving complex multi-repository issues |
| **2026-08-13** | **DeepSeek-V4 Pro** | DeepSeek AI | MoE (671B MLA) | 78.4% | \$0.66 | Highest open-weights score recorded on SWE-bench Pro |
| **2026-08-16** | GLM-5.3 | Zhipu AI | Agentic MoE | 66.5% | \$1.92 | Specialized agentic CLI and tool execution framework |
| **2026-09-02** | Qwen 3.8 Max | Alibaba Cloud | MoE Enterprise | 74.8% | \$6.00 | Alibaba Cloud flagship enterprise foundation model |
| **2026-09-10** | **DeepSeek-V4.1 Flash** | DeepSeek AI | MoE (236B/21B MLA) | 72.5% | \$0.28 | Pareto champion: 72.5% resolution at only \$0.28/Mtok |
| **2026-09-15** | GPT-6 Astra | OpenAI | Reasoning Flagship | 81.5% | \$50.00 | OpenAI deep reasoning frontier flagship |
| **2026-09-21** | Claude Opus 5.5 | Anthropic | Frontier Flagship | 89.9% | \$20.00 | Current SOTA ceiling: 89.9% resolution on SWE-bench Pro |


In [ ]:
try:
    import plotly.express as px
    import plotly.graph_objects as go
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "plotly"])
    import plotly.express as px
    import plotly.graph_objects as go

# --- 1. Pure Calculation Functions ---

def compute_agentic_task_cost(
    prompt_rate: float,
    completion_rate: float,
    cached_prompt_rate: float = 0.0,
    prompt_tokens: int = 100_000,
    completion_tokens: int = 22_000,
    cache_hit_rate: float = 0.70
) -> float:
    """Calculates representative agentic task cost (Workload W6: SWE-bench trajectory)."""
    effective_prompt_cost = (
        prompt_tokens * (1.0 - cache_hit_rate) * prompt_rate +
        prompt_tokens * cache_hit_rate * cached_prompt_rate
    )
    effective_completion_cost = completion_tokens * completion_rate
    return round((effective_prompt_cost + effective_completion_cost) / 1e6, 2)

def extract_pareto_frontier(df: pd.DataFrame, x_col: str = "cost_mtok", y_col: str = "score") -> pd.DataFrame:
    """Extracts the non-dominated Pareto frontier skyline (minimize x_col, maximize y_col)."""
    sorted_df = df.sort_values(by=[x_col, y_col], ascending=[True, False]).reset_index(drop=True)
    pareto_indices = []
    max_y = -float("inf")
    for idx, row in sorted_df.iterrows():
        if row[y_col] > max_y:
            pareto_indices.append(idx)
            max_y = row[y_col]
    return sorted_df.loc[pareto_indices].sort_values(x_col)

def extract_temporal_frontier(df: pd.DataFrame, date_col: str = "release_date", score_col: str = "score") -> pd.DataFrame:
    """Computes cumulative historical maximum (SOTA expansion frontier over time)."""
    sorted_df = df.sort_values(date_col).reset_index(drop=True)
    sota_rows = []
    max_score = -float("inf")
    for idx, row in sorted_df.iterrows():
        if row[score_col] > max_score:
            sota_rows.append(row)
            max_score = row[score_col]
    return pd.DataFrame(sota_rows)

# --- 2. Functional Data Extraction Pipelines ---

def extract_api_pareto_records(df_api: pd.DataFrame, df_bench: pd.DataFrame, benchmark: str = "SWE-bench Pro") -> pd.DataFrame:
    """Functional join of API rate cards and standardized evaluations for a target benchmark."""
    bench_filter = df_bench["benchmark"] == benchmark
    bench_df = (
        df_bench[bench_filter][["model_key", "score", "run_cost_usd"]]
        .rename(columns={"run_cost_usd": "benchmark_run_cost"})
    )
    
    regime_mapping = {
        "frontier-lab-first-party": "Proprietary Frontier API",
        "open-lab-first-party": "Hosted Open-Weight API"
    }
    
    return (
        df_api
        .merge(bench_df, on="model_key", how="inner")
        .assign(
            name=lambda df: df["model_id"],
            benchmark=benchmark,
            score=lambda df: df["score"],
            regime=lambda df: df["venue_type"].map(lambda v: regime_mapping.get(v, "Hosted Open-Weight API")),
            platform=lambda df: df["provider"].str.title() + " API",
            precision=lambda df: df["quantization_served"].str.upper(),
            cost_mtok=lambda df: df["output_usd_mtok"],
            task_cost=lambda df: df.apply(
                lambda row: compute_agentic_task_cost(
                    prompt_rate=row["input_usd_mtok"],
                    completion_rate=row["output_usd_mtok"],
                    cached_prompt_rate=row["cached_input_usd_mtok"]
                ), 
                axis=1
            ),
            tps=lambda df: df["throughput_tps"],
            release_date=lambda df: pd.to_datetime(df["release_date"]),
            lab=lambda df: df["lab_origin"] if "lab_origin" in df.columns else "Unknown",
            params=lambda df: df["params_display"] if "params_display" in df.columns else "N/A",
            arch=lambda df: df["arch_structure"] if "arch_structure" in df.columns else "Dense",
            attn=lambda df: df["attention_mech"] if "attention_mech" in df.columns else "GQA"
        )
        [["name", "model_key", "benchmark", "score", "regime", "platform", "precision", "cost_mtok", "task_cost", "tps", "release_date", "lab", "params", "arch", "attn"]]
    )

def extract_self_hosted_records(df_chinese_hardware: pd.DataFrame, benchmark: str = "SWE-bench Pro") -> pd.DataFrame:
    """Functional derivation of self-hosted quantized models from empirical hardware matrix for target benchmark."""
    bench_col_map = {
        "SWE-bench Pro": "swe_bench_pro_pct",
        "Terminal-Bench 4.0": "terminal_bench_pct",
        "AA Intelligence Index": "aa_index_score"
    }
    score_col = bench_col_map.get(benchmark, "swe_bench_pro_pct")
    resolve_regime = lambda v: "Apple Silicon (Self-Hosted)" if "Apple" in str(v) else ("NVIDIA DGX Spark (Self-Hosted)" if "NVIDIA" in str(v) else "Self-Hosted Hardware")
    
    return (
        df_chinese_hardware
        .query(f"{score_col} > 0 and local_cost_out_mtok_50util > 0")
        .assign(
            name=lambda df: df["model"],
            model_key=lambda df: df["model"].str.lower().str.replace(" ", "-"),
            benchmark=benchmark,
            score=lambda df: df[score_col],
            regime=lambda df: df["vendor"].apply(resolve_regime),
            platform=lambda df: df["hardware_config"],
            precision=lambda df: df["quantization"],
            cost_mtok=lambda df: df["local_cost_out_mtok_50util"],
            task_cost=lambda df: df.apply(
                lambda row: compute_agentic_task_cost(
                    prompt_rate=row["local_cost_out_mtok_50util"] * 0.25,
                    completion_rate=row["local_cost_out_mtok_50util"],
                    cached_prompt_rate=0.0,
                    cache_hit_rate=0.0
                ),
                axis=1
            ),
            tps=lambda df: df["decode_tps"],
            release_date=pd.NaT,
            lab="Hardware Community",
            params=lambda df: df["quantization"],
            arch="Quantized",
            attn="Local"
        )
        [["name", "model_key", "benchmark", "score", "regime", "platform", "precision", "cost_mtok", "task_cost", "tps", "release_date", "lab", "params", "arch", "attn"]]
    )

def assemble_pareto_dataset(data_registry: dict, benchmark: str = "SWE-bench Pro") -> pd.DataFrame:
    """Composes datasets via pure functions into a unified Pareto DataFrame for target benchmark."""
    return pd.concat([
        extract_api_pareto_records(data_registry["api_prices"], data_registry["benchmarks"], benchmark),
        extract_self_hosted_records(data_registry["chinese_models"], benchmark)
    ], ignore_index=True).sort_values(["regime", "score"], ascending=[True, False])

# --- 3. Interactive Visualization Renderers --- 

def create_interactive_pareto_switcher(
    data_registry: dict,
    benchmarks: list = ["SWE-bench Pro", "Terminal-Bench 4.0", "AA Intelligence Index"]
) -> go.Figure:
    """Plot 1: Interactive Multi-Benchmark Pareto chart with interactive dropdown switcher."""
    palette = {
        "Proprietary Frontier API": "#f59e0b",
        "Hosted Open-Weight API": "#10b981",
        "Apple Silicon (Self-Hosted)": "#3b82f6",
        "NVIDIA DGX Spark (Self-Hosted)": "#84cc16"
    }
    regimes_order = [
        "Proprietary Frontier API", 
        "Hosted Open-Weight API", 
        "Apple Silicon (Self-Hosted)", 
        "NVIDIA DGX Spark (Self-Hosted)"
    ]
    
    bench_configs = {
        "SWE-bench Pro": {
            "label": "SWE-bench Pro v1.2",
            "title": "<b>Empirical Pareto Frontier: SWE-bench Pro vs. Economic Cost</b><br><sup>Software Engineering Repository Resolution (% Resolved)</sup>",
            "yaxis_title": "SWE-bench Pro v1.2 Resolution (%)",
            "yaxis_range": [50, 93],
            "ticksuffix": "%"
        },
        "Terminal-Bench 4.0": {
            "label": "Terminal-Bench 4.0",
            "title": "<b>Empirical Pareto Frontier: Terminal-Bench 4.0 vs. Economic Cost</b><br><sup>Command-Line &amp; Agentic System Tasks (% Tasks Completed)</sup>",
            "yaxis_title": "Terminal-Bench 4.0 Completion (%)",
            "yaxis_range": [55, 94],
            "ticksuffix": "%"
        },
        "AA Intelligence Index": {
            "label": "AA Intelligence Index",
            "title": "<b>Empirical Pareto Frontier: AA Intelligence Index vs. Economic Cost</b><br><sup>Artificial Analysis Standardized Quality Index Score</sup>",
            "yaxis_title": "AA Intelligence Index v4.3.2",
            "yaxis_range": [40, 60],
            "ticksuffix": " pts"
        }
    }
    
    fig = go.Figure()
    trace_bench_map = []
    
    for b_idx, b in enumerate(benchmarks):
        df_bench = assemble_pareto_dataset(data_registry, benchmark=b)
        is_active = (b_idx == 0)
        
        # 1. Scatter traces per regime
        for regime in regimes_order:
            sub = df_bench[df_bench["regime"] == regime]
            if len(sub) > 0:
                hover_txt = [
                    f"<b>{row['name']}</b><br>Platform: {row['platform']}<br>Precision: {row['precision']}<br>" +
                    f"Score: {row['score']:.1f}{bench_configs[b]['ticksuffix']}<br>Output Cost: ${row['cost_mtok']:.2f}/Mtok<br>Task Cost: ${row['task_cost']:.2f}<br>Speed: {row['tps']:.1f} tok/s"
                    for _, row in sub.iterrows()
                ]
                fig.add_trace(go.Scatter(
                    x=sub["cost_mtok"],
                    y=sub["score"],
                    mode="markers",
                    name=regime,
                    legendgroup=regime,
                    showlegend=is_active,
                    visible=is_active,
                    marker=dict(size=12, color=palette.get(regime, "#64748b"), line=dict(width=1, color="#0f172a")),
                    hovertext=hover_txt,
                    hoverinfo="text"
                ))
                trace_bench_map.append(b)
        
        # 2. Pareto frontier line trace
        df_frontier = extract_pareto_frontier(df_bench, x_col="cost_mtok", y_col="score")
        fig.add_trace(go.Scatter(
            x=df_frontier["cost_mtok"],
            y=df_frontier["score"],
            mode="lines+markers",
            name=f"Optimal Envelope ({bench_configs[b]['label']})",
            legendgroup="envelope",
            showlegend=is_active,
            visible=is_active,
            line=dict(color="#dc2626", width=2.5, dash="dash"),
            marker=dict(size=8, color="#dc2626", symbol="diamond"),
            hovertext=[
                f"<b>[Optimal Pareto] {row['name']}</b><br>Platform: {row['platform']}<br>Score: {row['score']:.1f}{bench_configs[b]['ticksuffix']}<br>Cost: ${row['cost_mtok']:.2f}/Mtok<br>Task Cost: ${row['task_cost']:.2f}"
                for _, row in df_frontier.iterrows()
            ],
            hoverinfo="text"
        ))
        trace_bench_map.append(b)
    
    # Build dropdown updatemenus
    buttons = []
    for b in benchmarks:
        vis = [t_b == b for t_b in trace_bench_map]
        cfg = bench_configs[b]
        buttons.append(dict(
            label=f"Benchmark: {cfg['label']}",
            method="update",
            args=[
                {"visible": vis},
                {
                    "title.text": cfg["title"],
                    "yaxis.title.text": cfg["yaxis_title"],
                    "yaxis.range": cfg["yaxis_range"],
                    "yaxis.ticksuffix": cfg["ticksuffix"]
                }
            ]
        ))
    
    fig.update_layout(
        updatemenus=[
            dict(
                active=0,
                buttons=buttons,
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.0,
                xanchor="left",
                y=1.20,
                yanchor="top",
                bgcolor="rgba(255, 255, 255, 0.95)",
                bordercolor="#cbd5e1",
                borderwidth=1,
                font=dict(size=12, color="#0f172a")
            )
        ],
        xaxis=dict(type="log", tickformat="$.2f", gridcolor="#f1f5f9", ticks="outside", showline=True, linecolor="#94a3b8", title="Effective Cost per Million Tokens (USD, Log Scale)"),
        yaxis=dict(range=[50, 93], ticksuffix="%", gridcolor="#f1f5f9", ticks="outside", showline=True, linecolor="#94a3b8", title="SWE-bench Pro v1.2 Resolution (%)"),
        title=bench_configs["SWE-bench Pro"]["title"],
        template="plotly_white",
        height=640,
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
            bgcolor="rgba(255, 255, 255, 0.85)", bordercolor="#e2e8f0", borderwidth=1
        )
    )
    return fig

def create_temporal_progression_plot(
    data_registry: dict,
    benchmarks: list = ["SWE-bench Pro", "Terminal-Bench 4.0", "AA Intelligence Index"]
) -> go.Figure:
    """Plot 2: Model Score vs. Release Date with interactive evaluation dropdown switcher."""
    palette = {
        "Proprietary Frontier API": "#f59e0b",
        "Hosted Open-Weight API": "#10b981"
    }
    regimes_order = ["Proprietary Frontier API", "Hosted Open-Weight API"]
    
    bench_configs = {
        "SWE-bench Pro": {
            "label": "SWE-bench Pro v1.2",
            "title": "<b>Frontier Velocity: SWE-bench Pro Score vs. Release Date</b><br><sup>Progression of Coding Intelligence across Frontier &amp; Open Labs (2024–2026)</sup>",
            "yaxis_title": "SWE-bench Pro v1.2 Resolution Rate (%)",
            "yaxis_range": [42, 93],
            "ticksuffix": "%"
        },
        "Terminal-Bench 4.0": {
            "label": "Terminal-Bench 4.0",
            "title": "<b>Frontier Velocity: Terminal-Bench 4.0 vs. Release Date</b><br><sup>Progression of Autonomous CLI &amp; System Agent Execution (2024–2026)</sup>",
            "yaxis_title": "Terminal-Bench 4.0 Tasks Completed (%)",
            "yaxis_range": [50, 93],
            "ticksuffix": "%"
        },
        "AA Intelligence Index": {
            "label": "AA Intelligence Index",
            "title": "<b>Frontier Velocity: AA Intelligence Index vs. Release Date</b><br><sup>Progression of Standardized Intelligence Index Score (2024–2026)</sup>",
            "yaxis_title": "AA Intelligence Index v4.3.2 (Score)",
            "yaxis_range": [40, 60],
            "ticksuffix": " pts"
        }
    }
    
    bench_annotations = {
        "SWE-bench Pro": [
            dict(
                x="2024-10-22", y=67.5,
                xref="x", yref="y",
                text="<b>Claude 3.5 Sonnet (v2)</b><br>10-Month SOTA Anchor",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#d97706",
                ax=-45, ay=-40,
                font=dict(size=10, color="#92400e"),
                bgcolor="rgba(254, 243, 199, 0.9)", bordercolor="#d97706", borderwidth=1
            ),
            dict(
                x="2024-12-26", y=59.8,
                xref="x", yref="y",
                text="<b>DeepSeek-V3</b><br>671B MLA MoE ($0.28/Mtok)",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#059669",
                ax=-35, ay=45,
                font=dict(size=10, color="#065f46"),
                bgcolor="rgba(209, 250, 229, 0.9)", bordercolor="#059669", borderwidth=1
            ),
            dict(
                x="2025-01-20", y=65.4,
                xref="x", yref="y",
                text="<b>DeepSeek-R1</b><br>Open-Weights RL Reasoning",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#059669",
                ax=-40, ay=-45,
                font=dict(size=10, color="#065f46"),
                bgcolor="rgba(209, 250, 229, 0.9)", bordercolor="#059669", borderwidth=1
            ),
            dict(
                x="2026-09-21", y=89.9,
                xref="x", yref="y",
                text="<b>Claude Opus 5.5</b><br>Current Frontier SOTA (89.9%)",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#dc2626",
                ax=-50, ay=35,
                font=dict(size=10, color="#991b1b"),
                bgcolor="rgba(254, 226, 226, 0.9)", bordercolor="#dc2626", borderwidth=1
            )
        ],
        "Terminal-Bench 4.0": [
            dict(
                x="2024-12-05", y=76.2,
                xref="x", yref="y",
                text="<b>OpenAI o1</b><br>Reasoning CLI Agent",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#d97706",
                ax=-40, ay=-40,
                font=dict(size=10, color="#92400e"),
                bgcolor="rgba(254, 243, 199, 0.9)", bordercolor="#d97706", borderwidth=1
            ),
            dict(
                x="2025-01-20", y=71.0,
                xref="x", yref="y",
                text="<b>DeepSeek-R1</b><br>Open CLI Reasoning (71.0%)",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#059669",
                ax=-40, ay=40,
                font=dict(size=10, color="#065f46"),
                bgcolor="rgba(209, 250, 229, 0.9)", bordercolor="#059669", borderwidth=1
            ),
            dict(
                x="2026-08-13", y=81.2,
                xref="x", yref="y",
                text="<b>DeepSeek-V4 Pro</b><br>Top Open-Weights CLI (81.2%)",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#059669",
                ax=-45, ay=-40,
                font=dict(size=10, color="#065f46"),
                bgcolor="rgba(209, 250, 229, 0.9)", bordercolor="#059669", borderwidth=1
            ),
            dict(
                x="2026-09-21", y=88.4,
                xref="x", yref="y",
                text="<b>Claude Opus 5.5</b><br>SOTA CLI Agent (88.4%)",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#dc2626",
                ax=-45, ay=35,
                font=dict(size=10, color="#991b1b"),
                bgcolor="rgba(254, 226, 226, 0.9)", bordercolor="#dc2626", borderwidth=1
            )
        ],
        "AA Intelligence Index": [
            dict(
                x="2024-12-05", y=51.5,
                xref="x", yref="y",
                text="<b>OpenAI o1</b><br>Test-Time Compute Benchmark",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#d97706",
                ax=-40, ay=-40,
                font=dict(size=10, color="#92400e"),
                bgcolor="rgba(254, 243, 199, 0.9)", bordercolor="#d97706", borderwidth=1
            ),
            dict(
                x="2025-01-20", y=49.5,
                xref="x", yref="y",
                text="<b>DeepSeek-R1</b><br>Open RL Baseline (49.5 pts)",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#059669",
                ax=-40, ay=40,
                font=dict(size=10, color="#065f46"),
                bgcolor="rgba(209, 250, 229, 0.9)", bordercolor="#059669", borderwidth=1
            ),
            dict(
                x="2026-08-13", y=54.2,
                xref="x", yref="y",
                text="<b>DeepSeek-V4 Pro</b><br>Open-Weights Frontier (54.2 pts)",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#059669",
                ax=-45, ay=-35,
                font=dict(size=10, color="#065f46"),
                bgcolor="rgba(209, 250, 229, 0.9)", bordercolor="#059669", borderwidth=1
            ),
            dict(
                x="2026-09-21", y=57.6,
                xref="x", yref="y",
                text="<b>Claude Opus 5.5</b><br>#1 Global Quality (57.6 pts)",
                showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="#dc2626",
                ax=-45, ay=35,
                font=dict(size=10, color="#991b1b"),
                bgcolor="rgba(254, 226, 226, 0.9)", bordercolor="#dc2626", borderwidth=1
            )
        ]
    }
    
    fig = go.Figure()
    trace_bench_map = []
    
    for b_idx, b in enumerate(benchmarks):
        is_active = (b_idx == 0)
        df_api_records = extract_api_pareto_records(data_registry["api_prices"], data_registry["benchmarks"], benchmark=b)
        df_clean = df_api_records.dropna(subset=["release_date"]).sort_values("release_date").copy()
        
        # 1. Scatter traces per regime
        for regime in regimes_order:
            sub = df_clean[df_clean["regime"] == regime]
            if len(sub) > 0:
                hover_txt = [
                    f"<b>{row['name']}</b><br>Lab: {row.get('lab', 'Unknown')}<br>Platform: {row['platform']}<br>Released: {row['release_date'].strftime('%b %d, %Y')}<br>" +
                    f"Score: {row['score']:.1f}{bench_configs[b]['ticksuffix']}<br>Output Cost: ${row['cost_mtok']:.2f}/Mtok<br>Task Cost: ${row['task_cost']:.2f}<br>Speed: {row['tps']:.1f} tok/s"
                    for _, row in sub.iterrows()
                ]
                fig.add_trace(go.Scatter(
                    x=sub["release_date"],
                    y=sub["score"],
                    mode="markers",
                    name=regime,
                    legendgroup=regime,
                    showlegend=is_active,
                    visible=is_active,
                    marker=dict(size=13, color=palette.get(regime, "#64748b"), line=dict(width=1, color="#1e293b")),
                    hovertext=hover_txt,
                    hoverinfo="text"
                ))
                trace_bench_map.append(b)
        
        # 2. SOTA capability envelope
        df_sota = extract_temporal_frontier(df_clean, date_col="release_date", score_col="score")
        hover_sota = [
            f"<b>[SOTA Milestone] {row['name']}</b><br>Lab: {row.get('lab', 'Unknown')}<br>Platform: {row['platform']}<br>Score: {row['score']:.1f}{bench_configs[b]['ticksuffix']}<br>Cost: ${row['cost_mtok']:.2f}/Mtok"
            for _, row in df_sota.iterrows()
        ]
        fig.add_trace(go.Scatter(
            x=df_sota["release_date"],
            y=df_sota["score"],
            mode="lines+markers",
            name=f"SOTA Envelope ({bench_configs[b]['label']})",
            legendgroup="sota",
            showlegend=is_active,
            visible=is_active,
            line=dict(color="#dc2626", width=2.5, dash="dash"),
            marker=dict(size=8, color="#dc2626", symbol="diamond"),
            hovertext=hover_sota,
            hoverinfo="text"
        ))
        trace_bench_map.append(b)
    
    # Dropdown buttons
    buttons = []
    for b in benchmarks:
        vis = [t_b == b for t_b in trace_bench_map]
        cfg = bench_configs[b]
        buttons.append(dict(
            label=f"Benchmark: {cfg['label']}",
            method="update",
            args=[
                {"visible": vis},
                {
                    "title.text": cfg["title"],
                    "yaxis.title.text": cfg["yaxis_title"],
                    "yaxis.range": cfg["yaxis_range"],
                    "yaxis.ticksuffix": cfg["ticksuffix"],
                    "annotations": bench_annotations[b]
                }
            ]
        ))
    
    fig.update_layout(
        updatemenus=[
            dict(
                active=0,
                buttons=buttons,
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.0,
                xanchor="left",
                y=1.20,
                yanchor="top",
                bgcolor="rgba(255, 255, 255, 0.95)",
                bordercolor="#cbd5e1",
                borderwidth=1,
                font=dict(size=12, color="#0f172a")
            )
        ],
        xaxis=dict(gridcolor="#f1f5f9", ticks="outside", showline=True, linecolor="#94a3b8", title="Official Release Date"),
        yaxis=dict(
            range=bench_configs["SWE-bench Pro"]["yaxis_range"],
            ticksuffix=bench_configs["SWE-bench Pro"]["ticksuffix"],
            gridcolor="#f1f5f9",
            ticks="outside",
            showline=True,
            linecolor="#94a3b8",
            title=bench_configs["SWE-bench Pro"]["yaxis_title"]
        ),
        title=bench_configs["SWE-bench Pro"]["title"],
        template="plotly_white",
        height=650,
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
            bgcolor="rgba(255, 255, 255, 0.85)", bordercolor="#e2e8f0", borderwidth=1
        ),
        annotations=bench_annotations["SWE-bench Pro"]
    )
    return fig

# Execute Plot 1 and Plot 2 pipelines
print("Generating Plot 1: Multi-Benchmark Interactive Pareto Switcher...")
fig_pareto = create_interactive_pareto_switcher(datasets)
fig_pareto.show()

print("Generating Plot 2: Frontier Velocity (Score vs. Release Date with Multi-Eval Dropdown)...")
fig_temporal = create_temporal_progression_plot(datasets)
fig_temporal.show()


### 6. Interactive Model Catalogue & Architectural Register
A multi-dimensional catalogue registering all 36 evaluated frontier and landmark models across primary architectural, organizational, and economic vectors:

* **Organizational & Licensing Origin**: Lab/Creator (Anthropic, OpenAI, Google, DeepSeek AI, Moonshot AI, Alibaba Cloud, Meta AI, Zhipu AI, ByteDance, 01.AI, StepFun, MiniMax, xAI) and Open-Weights vs. Proprietary API status.
* **Architectural Specifications**: Parameter scale (resident total vs. active per-token parameters for fine-grained MoE), dense vs. sparse mixture-of-experts routing, and attention mechanism (**Multi-Head Latent Attention `MLA`** vs. **Grouped Query Attention `GQA`**).
* **Cross-Benchmark Evaluation**: Side-by-side performance on **SWE-bench Pro v1.2** (repository-level software engineering), **Terminal-Bench 4.0** (autonomous CLI execution), and **Artificial Analysis Intelligence Index v4.3.2** (standardized composite reasoning).
* **Inference Token Economics**: Blended input and output token rate card (\$/Mtok) and maximum supported native context window.

#### Interactive Sorting & Slicing Modes:
1. **Sortable HTML Data Table**: Click **ANY column header** to sort ascending (▲) or descending (▼) with dynamic numeric/date type handling. Use the live search box to filter models or labs instantly, and select cohort slices from the dropdown.
2. **Plotly Architectural Catalogue**: Standard Plotly `go.Table` visualization with dropdown cohort switching.


In [ ]:
from IPython.display import HTML, display

def build_catalogue_dataframe(data_registry: dict) -> pd.DataFrame:
    """Extracts and joins complete metadata, multi-benchmark evaluations, and token pricing."""
    df_api = data_registry["api_prices"].copy()
    df_b = data_registry["benchmarks"].copy()
    
    # Pivot benchmark evaluations to extract model scores across evals
    swe_map = df_b[df_b["benchmark"] == "SWE-bench Pro"].set_index("model_key")["score"].to_dict()
    tb_map = df_b[df_b["benchmark"] == "Terminal-Bench 4.0"].set_index("model_key")["score"].to_dict()
    aa_map = df_b[df_b["benchmark"] == "AA Intelligence Index"].set_index("model_key")["score"].to_dict()
    
    chinese_labs = {
        "DeepSeek AI", "Moonshot AI", "Alibaba Cloud", "Zhipu AI", 
        "ByteDance", "01.AI", "StepFun", "MiniMax"
    }
    
    records = []
    for _, row in df_api.iterrows():
        mkey = row["model_key"]
        lab = row.get("lab_origin", "Unknown")
        is_open = str(row["open_weight"]).lower() == "true"
        arch = row.get("arch_structure", "Dense")
        attn = row.get("attention_mech", "GQA")
        params = row.get("params_display", "N/A")
        rel_date = str(row["release_date"])
        swe_score = swe_map.get(mkey, np.nan)
        tb_score = tb_map.get(mkey, np.nan)
        aa_score = aa_map.get(mkey, np.nan)
        in_rate = row["input_usd_mtok"]
        out_rate = row["output_usd_mtok"]
        
        records.append({
            "model_key": mkey,
            "name": row["model_id"],
            "lab": lab,
            "is_chinese": lab in chinese_labs,
            "open_weight": is_open,
            "weights_label": "Open-Weights" if is_open else "Proprietary",
            "release_date": rel_date,
            "params": params,
            "arch": arch,
            "attn": attn,
            "swe_bench": swe_score,
            "swe_bench_str": f"{swe_score:.1f}%" if pd.notnull(swe_score) else "—",
            "terminal_bench": tb_score,
            "terminal_bench_str": f"{tb_score:.1f}%" if pd.notnull(tb_score) else "—",
            "aa_index": aa_score,
            "aa_index_str": f"{aa_score:.1f}" if pd.notnull(aa_score) else "—",
            "price_str": f"${in_rate:.2f} / ${out_rate:.2f}",
            "out_usd_mtok": out_rate,
            "context_window": row["context_window"]
        })
    
    return pd.DataFrame(records).sort_values("release_date", ascending=False).reset_index(drop=True)

def generate_sortable_catalogue_html(df_cat: pd.DataFrame) -> str:
    """Generates an interactive HTML table where clicking ANY column header sorts the rows."""
    cohorts = [
        ("all", "All Models (36)"),
        ("chinese", "Chinese Labs (18)"),
        ("open", "Open-Weights Only (14)"),
        ("closed", "Proprietary Only (22)"),
        ("moe", "MoE Architectures (19)"),
        ("dense", "Dense Architectures (17)")
    ]
    
    opt_html = "".join([f"<option value='{cid}'>{cname}</option>" for cid, cname in cohorts])
    
    rows_html = []
    for idx, r in df_cat.iterrows():
        bg = "#f8fafc" if idx % 2 == 0 else "#ffffff"
        ch_flag = "1" if r["is_chinese"] else "0"
        op_flag = "1" if r["open_weight"] else "0"
        arch_val = r["arch"]
        attn_val = r["attn"]
        swe_num = r["swe_bench"] if pd.notnull(r["swe_bench"]) else -1
        tb_num = r["terminal_bench"] if pd.notnull(r["terminal_bench"]) else -1
        aa_num = r["aa_index"] if pd.notnull(r["aa_index"]) else -1
        price_num = r["out_usd_mtok"]
        
        w_bg = "#ecfdf5" if r["open_weight"] else "#f1f5f9"
        w_color = "#047857" if r["open_weight"] else "#475569"
        w_badge = f"<span style='background: {w_bg}; color: {w_color}; padding: 2px 7px; border-radius: 9999px; font-weight: 600; font-size: 11px;'>{r['weights_label']}</span>"
        
        arch_bg = "#eff6ff" if arch_val == "MoE" else "#f8fafc"
        arch_color = "#1d4ed8" if arch_val == "MoE" else "#334155"
        arch_badge = f"<span style='background: {arch_bg}; color: {arch_color}; padding: 2px 6px; border-radius: 4px; font-weight: 500;'>{arch_val}</span>"
        
        attn_bg = "#fef3c7" if attn_val == "MLA" else "#f1f5f9"
        attn_color = "#b45309" if attn_val == "MLA" else "#475569"
        attn_badge = f"<span style='background: {attn_bg}; color: {attn_color}; padding: 2px 6px; border-radius: 4px; font-weight: 600;'>{attn_val}</span>"
        
        row_str = (
            f"<tr style='background: {bg}; border-bottom: 1px solid #f1f5f9;' onmouseover=\"this.style.background='#f1f5f9'\" onmouseout=\"this.style.background='{bg}'\" data-chinese='{ch_flag}' data-open='{op_flag}' data-arch='{arch_val}' data-attn='{attn_val}'>" +
            f"<td style='padding: 8px 12px; font-weight: 600; color: #0f172a;' data-val='{r['model_key']}'>{r['model_key']}</td>" +
            f"<td style='padding: 8px 12px; color: #334155;' data-val='{r['lab']}'>{r['lab']}</td>" +
            f"<td style='padding: 8px 10px; text-align: center;' data-val='{r['weights_label']}'>{w_badge}</td>" +
            f"<td style='padding: 8px 10px; text-align: center; color: #475569;' data-val='{r['release_date']}'>{r['release_date']}</td>" +
            f"<td style='padding: 8px 10px; text-align: center; color: #334155;' data-val='{r['params']}'>{r['params']}</td>" +
            f"<td style='padding: 8px 8px; text-align: center;' data-val='{arch_val}'>{arch_badge}</td>" +
            f"<td style='padding: 8px 8px; text-align: center;' data-val='{attn_val}'>{attn_badge}</td>" +
            f"<td style='padding: 8px 10px; text-align: center; font-weight: 600; color: #0f172a;' data-val='{swe_num}'>{r['swe_bench_str']}</td>" +
            f"<td style='padding: 8px 10px; text-align: center; font-weight: 600; color: #0f172a;' data-val='{tb_num}'>{r['terminal_bench_str']}</td>" +
            f"<td style='padding: 8px 10px; text-align: center; font-weight: 600; color: #0f172a;' data-val='{aa_num}'>{r['aa_index_str']}</td>" +
            f"<td style='padding: 8px 12px; text-align: right; font-weight: 600; color: #047857;' data-val='{price_num}'>${price_num:.2f}</td>" +
            "</tr>"
        )
        rows_html.append(row_str)
    
    table_body = "\n".join(rows_html)
    
    template = """
<div id='cat-widget-container' style='font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; background: #ffffff; border: 1px solid #e2e8f0; border-radius: 12px; padding: 18px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.05); margin: 20px 0;'>
    <div style='display: flex; justify-content: space-between; align-items: flex-start; flex-wrap: wrap; gap: 12px; margin-bottom: 14px;'>
        <div>
            <h3 style='margin: 0 0 4px 0; font-size: 17px; font-weight: 700; color: #0f172a;'>Frontier Model Catalogue & Architectural Register</h3>
            <p style='margin: 0; font-size: 12px; color: #64748b;'>Click <b>any column header</b> to sort ascending/descending. Filter by cohort or search attributes.</p>
        </div>
        <div style='display: flex; gap: 8px; align-items: center; flex-wrap: wrap;'>
            <input id='cat-search-box' type='text' placeholder='🔍 Search models, labs...' style='padding: 6px 10px; font-size: 12px; border: 1px solid #cbd5e1; border-radius: 6px; width: 180px; outline: none;' onkeyup='filterCatTable()'>
            <select id='cat-cohort-select' style='padding: 6px 10px; font-size: 12px; border: 1px solid #cbd5e1; border-radius: 6px; background: #ffffff; color: #0f172a; cursor: pointer; outline: none;' onchange='filterCatTable()'>
                __OPT_HTML__
            </select>
        </div>
    </div>
    <div style='overflow-x: auto; max-height: 520px; border: 1px solid #e2e8f0; border-radius: 8px;'>
        <table id='cat-sortable-table' style='width: 100%; border-collapse: collapse; font-size: 12px; text-align: left;'>
            <thead style='position: sticky; top: 0; background: #1e293b; color: #f8fafc; z-index: 10;'>
                <tr>
                    <th onclick="sortCatTable(0, 'str')" style='padding: 10px 12px; cursor: pointer; user-select: none; white-space: nowrap;'>Model Identifier <span id='icon-0'>↕</span></th>
                    <th onclick="sortCatTable(1, 'str')" style='padding: 10px 12px; cursor: pointer; user-select: none; white-space: nowrap;'>Lab / Org <span id='icon-1'>↕</span></th>
                    <th onclick="sortCatTable(2, 'str')" style='padding: 10px 8px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>Weights <span id='icon-2'>↕</span></th>
                    <th onclick="sortCatTable(3, 'date')" style='padding: 10px 8px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>Release Date <span id='icon-3'>↕</span></th>
                    <th onclick="sortCatTable(4, 'str')" style='padding: 10px 8px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>Parameters <span id='icon-4'>↕</span></th>
                    <th onclick="sortCatTable(5, 'str')" style='padding: 10px 6px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>Arch <span id='icon-5'>↕</span></th>
                    <th onclick="sortCatTable(6, 'str')" style='padding: 10px 6px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>Attn <span id='icon-6'>↕</span></th>
                    <th onclick="sortCatTable(7, 'num')" style='padding: 10px 8px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>SWE-bench Pro <span id='icon-7'>↕</span></th>
                    <th onclick="sortCatTable(8, 'num')" style='padding: 10px 8px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>Terminal-Bench <span id='icon-8'>↕</span></th>
                    <th onclick="sortCatTable(9, 'num')" style='padding: 10px 8px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>AA Index <span id='icon-9'>↕</span></th>
                    <th onclick="sortCatTable(10, 'num')" style='padding: 10px 12px; cursor: pointer; user-select: none; text-align: right; white-space: nowrap;'>Out ($/Mtok) <span id='icon-10'>↕</span></th>
                </tr>
            </thead>
            <tbody id='cat-tbody'>
                __TABLE_BODY__
            </tbody>
        </table>
    </div>
    <div style='display: flex; justify-content: space-between; align-items: center; margin-top: 10px; font-size: 11px; color: #64748b;'>
        <div id='cat-status-text'>Showing 36 of 36 models</div>
        <div>💡 Click headers like <b>SWE-bench Pro</b> or <b>Out ($/Mtok)</b> to sort highest/lowest.</div>
    </div>
</div>
<script>
var curCol = -1; var curAsc = false;
function sortCatTable(idx, typ) {
    var tb = document.getElementById('cat-tbody'); if (!tb) return;
    var rows = Array.from(tb.querySelectorAll('tr'));
    if (curCol === idx) { curAsc = !curAsc; } else { curCol = idx; curAsc = (typ === 'str' || typ === 'date'); }
    for (var i = 0; i <= 10; i++) {
        var ic = document.getElementById('icon-' + i);
        if (ic) { ic.innerHTML = (i === idx) ? (curAsc ? '▲' : '▼') : '↕'; ic.style.color = (i === idx) ? '#38bdf8' : '#94a3b8'; }
    }
    rows.sort(function(a, b) {
        var vA = a.children[idx].getAttribute('data-val') || a.children[idx].innerText.trim();
        var vB = b.children[idx].getAttribute('data-val') || b.children[idx].innerText.trim();
        if (typ === 'num') {
            var nA = parseFloat(vA); var nB = parseFloat(vB);
            if (isNaN(nA)) nA = -999999;
            if (isNaN(nB)) nB = -999999;
            return curAsc ? (nA - nB) : (nB - nA);
        } else {
            return curAsc ? vA.localeCompare(vB) : vB.localeCompare(vA);
        }
    });
    rows.forEach(function(r) { tb.appendChild(r); });
}
function filterCatTable() {
    var q = (document.getElementById('cat-search-box').value || '').toLowerCase();
    var coh = (document.getElementById('cat-cohort-select').value || 'all');
    var tb = document.getElementById('cat-tbody'); if (!tb) return;
    var rows = tb.querySelectorAll('tr'); var count = 0;
    rows.forEach(function(r) {
        var txt = r.innerText.toLowerCase();
        var mQ = txt.indexOf(q) !== -1;
        var isCh = r.getAttribute('data-chinese') === '1';
        var isOp = r.getAttribute('data-open') === '1';
        var arch = r.getAttribute('data-arch');
        var mC = true;
        if (coh === 'chinese') mC = isCh;
        else if (coh === 'open') mC = isOp;
        else if (coh === 'closed') mC = !isOp;
        else if (coh === 'moe') mC = (arch === 'MoE');
        else if (coh === 'dense') mC = (arch === 'Dense');
        if (mQ && mC) { r.style.display = ''; count++; } else { r.style.display = 'none'; }
    });
    var st = document.getElementById('cat-status-text'); if (st) st.innerText = 'Showing ' + count + ' of ' + rows.length + ' models';
}
</script>
"""
    return template.replace("__OPT_HTML__", opt_html).replace("__TABLE_BODY__", table_body)

def display_sortable_model_catalogue(data_registry: dict):
    """Displays the interactive sortable HTML catalogue table."""
    df_cat = build_catalogue_dataframe(data_registry)
    html_widget = generate_sortable_catalogue_html(df_cat)
    display(HTML(html_widget))

def create_interactive_model_catalogue(data_registry: dict) -> go.Figure:
    """Renders an interactive, filterable model catalogue table using Plotly updatemenus."""
    df_cat = build_catalogue_dataframe(data_registry)
    
    cohorts = [
        ("All Models (36)", df_cat),
        ("Chinese Labs (18)", df_cat[df_cat["is_chinese"]]),
        ("Open-Weights (14)", df_cat[df_cat["open_weight"]]),
        ("Frontier Proprietary (22)", df_cat[~df_cat["open_weight"]]),
        ("MoE Architectures (19)", df_cat[df_cat["arch"] == "MoE"]),
        ("Dense Architectures (17)", df_cat[df_cat["arch"] == "Dense"])
    ]
    
    fig = go.Figure()
    
    for c_idx, (name, df_sub) in enumerate(cohorts):
        is_active = (c_idx == 0)
        n_rows = len(df_sub)
        fill_colors = ["#f8fafc" if i % 2 == 0 else "#ffffff" for i in range(n_rows)]
        
        fig.add_trace(go.Table(
            name=name,
            visible=is_active,
            header=dict(
                values=[
                    "<b>Model Identifier</b>", "<b>Lab / Org</b>", "<b>Weights</b>", 
                    "<b>Release Date</b>", "<b>Parameters</b>", "<b>Arch</b>", "<b>Attn</b>",
                    "<b>SWE-bench Pro</b>", "<b>Terminal-Bench</b>", "<b>AA Index</b>", "<b>In / Out ($/Mtok)</b>"
                ],
                fill_color="#1e293b",
                font=dict(color="#f8fafc", size=11, family="Inter, -apple-system, sans-serif"),
                align=["left", "left", "center", "center", "center", "center", "center", "center", "center", "center", "right"],
                height=32
            ),
            cells=dict(
                values=[
                    df_sub["model_key"],
                    df_sub["lab"],
                    df_sub["weights_label"],
                    df_sub["release_date"],
                    df_sub["params"],
                    df_sub["arch"],
                    df_sub["attn"],
                    df_sub["swe_bench_str"],
                    df_sub["terminal_bench_str"],
                    df_sub["aa_index_str"],
                    df_sub["price_str"]
                ],
                fill_color=[fill_colors] * 11,
                font=dict(color="#0f172a", size=11, family="Inter, -apple-system, sans-serif"),
                align=["left", "left", "center", "center", "center", "center", "center", "center", "center", "center", "right"],
                height=26
            )
        ))
    
    buttons = []
    for c_idx, (name, df_sub) in enumerate(cohorts):
        vis = [i == c_idx for i in range(len(cohorts))]
        buttons.append(dict(
            label=f"Cohort: {name}",
            method="update",
            args=[
                {"visible": vis},
                {"title.text": f"<b>Frontier Model Catalogue &amp; Architectural Register</b><br><sup>Filter: {name} ({len(df_sub)} models) | Lab Origin, Architecture, Multi-Eval Scores &amp; Token Pricing</sup>"}
            ]
        ))
    
    fig.update_layout(
        updatemenus=[
            dict(
                active=0,
                buttons=buttons,
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.0,
                xanchor="left",
                y=1.18,
                yanchor="top",
                bgcolor="rgba(255, 255, 255, 0.95)",
                bordercolor="#cbd5e1",
                borderwidth=1,
                font=dict(size=12, color="#0f172a")
            )
        ],
        title="<b>Frontier Model Catalogue &amp; Architectural Register</b><br><sup>Filter: All Models (36) | Lab Origin, Architecture, Multi-Eval Scores &amp; Token Pricing</sup>",
        template="plotly_white",
        height=660,
        margin=dict(l=10, r=10, t=80, b=10)
    )
    return fig

# Render interactive sortable HTML catalogue
print("Rendering Sortable Model Catalogue (Click Any Column Header to Sort)...")
display_sortable_model_catalogue(datasets)

# Render Plotly figure
print("\nRendering Plotly Model Catalogue...")
fig_catalogue = create_interactive_model_catalogue(datasets)
fig_catalogue.show()


### 7. Open-Weights Hardware Residency & RAM Sizing Matrix (1TB, 512GB, 256GB, 128GB, 64GB, 32GB)
A rigorous physical sizing matrix quantifying total system memory (RAM / VRAM) required to load, initialize, and execute open-weights models across standard quantization precisions and context lengths.

#### A. The Physical Sizing Formulation
$$\text{Total Memory Footprint (GB)} = \text{Total Weights (GB)} + \text{KV Cache Footprint (GB)} + \text{Runtime/CUDA Overhead (6 GB)}$$
$$\text{Weights (GB)} = \text{Total Parameters (Billions)} \times \frac{\text{Quantization Bits}}{8} \times 1.05\text{ (Packing & Metadata Margin)}$$

* **Weight Footprint across Quantizations**:
  * **FP16 / BF16 (16-bit)**: $\approx 2.05\text{ GB per billion parameters}$ (e.g., DeepSeek 671B requires $1,375\text{ GB}$ weights alone; 70B requires $141\text{ GB}$).
  * **FP8 / INT8 (8-bit)**: $\approx 1.05\text{ GB per billion parameters}$ (e.g., DeepSeek 671B requires $705\text{ GB}$; 70B requires $74\text{ GB}$).
  * **Q5_K_M (5.5-bit)**: $\approx 0.72\text{ GB per billion parameters}$ (e.g., DeepSeek 236B requires $164\text{ GB}$; 70B requires $51\text{ GB}$).
  * **Q4_K_M / NVFP4 (4.5-bit / 4.0-bit)**: $\approx 0.55\text{–}0.56\text{ GB per billion parameters}$ (e.g., DeepSeek 685B requires $392\text{ GB}$; 236B requires $126\text{ GB}$; 70B requires $40\text{ GB}$).
  * **Q3_K_M / UD-Q3 (3.3-bit / 3.5-bit)**: $\approx 0.44\text{ GB per billion parameters}$ (e.g., DeepSeek 685B requires $288\text{ GB}$; 236B requires $98\text{ GB}$).
* **KV Cache Sizing at 32k Tokens**:
  * **Multi-Head Latent Attention (MLA)**: DeepSeek V3/V4 compresses key-value states down to $68.6\text{ KB/token}$, requiring only **$2.25\text{ GB}$** at $32\text{k}$ context (or **$1.12\text{ GB}$** in FP8).
  * **Grouped Query Attention (GQA)**: Llama 3 70B / 405B scales to $320\text{ KB/token}$, consuming **$10.5\text{ GB}$** at $32\text{k}$ context.

---

#### B. Hardware Memory Tier Classifications
* **$\ge$ 1 TB Tier** (*Datacenter Scale*: NVIDIA HGX H200 8-GPU 1.12TB HBM3e, Multi-Node Clusters):
  Can host 685B / 671B models in uncompressed **FP8 (728 GB)** or 405B in **FP16 (846 GB)** with massive multi-user batch concurrency.
* **512 GB Tier** (*Single-Box Desktop Flagship*: Apple Mac Studio M5 Ultra 512GB Unified Memory, Dual 256GB Clusters):
  The only desktop tier capable of running trillion-class MoE weights (**DeepSeek-V4 Pro 685B at 402 GB** or **DeepSeek-V3/R1 671B at 385 GB**) in **Q4_K_M** with zero expert paging. Can also host Llama 3.1 405B in Q8 / Q4.
* **256 GB Tier** (*Workstation Sweet Spot*: Apple Mac Studio M5 Ultra 256GB, Dual DGX Spark 256GB Cluster):
  Sweet spot for **236B MoE (DeepSeek V4.1 Flash at 134 GB resident)**, achieving **$48.2\text{ tok/s}$** interactive decode speeds. Easily clears 70B/72B in uncompressed FP16 ($155\text{ GB}$) and fits Qwen 3.5-397B in Q4 ($238\text{ GB}$).
* **128 GB Tier** (*Compact Workstation*: NVIDIA DGX Spark 128GB GB10, Mac Studio M5 Max 128GB):
  Hits the Residency Wall for 236B models (134 GB total exceeds 128 GB memory boundary). **Sweet spot for 70B / 72B dense models in Q8 ($90\text{ GB}$) or Q4 ($56\text{ GB}$)** and 27B–35B in uncompressed BF16.
* **64 GB Tier** (*Pro Workstation*: Dual 24GB GPUs [48GB], Mac Studio Max 64GB):
  Runs **70B/72B models in Q4_K_M ($56\text{ GB}$ resident)** under tight context bounds, and runs 32B–35B models in Q8 ($41\text{–}45\text{ GB}$) with full headroom.
* **32 GB Tier** (*Consumer Laptops & Single GPUs*: MacBook Pro M5 32GB, Single RTX 4090 24GB):
  Hits the Residency Wall for 70B+. Runs **27B–35B models in Q4 ($20\text{–}28\text{ GB}$)** and 7B–14B models in uncompressed FP16/BF16.

Use the interactive sortable table and residency wall chart below to inspect memory requirements across all open-weights models.

In [ ]:
def generate_sortable_ram_table_html(df_ram: pd.DataFrame) -> str:
    """Generates an interactive, sortable HTML table for open-weights hardware residency."""
    tier_filters = [
        ("all", "All Open-Weights Models (16)"),
        ("32gb", "Can Run on 32 GB (Laptops / Single 24GB GPU)"),
        ("64gb", "Can Run on 64 GB (Workstations / Dual GPUs)"),
        ("128gb", "Can Run on 128 GB (DGX Spark / Mac Studio Max)"),
        ("256gb", "Can Run on 256 GB (Mac Studio Ultra 256GB)"),
        ("512gb", "Can Run on 512 GB (Mac Studio Ultra 512GB)"),
        ("1tb", "Requires 1 TB+ Datacenter Scale")
    ]
    
    opt_html = "".join([f"<option value='{tid}'>{tname}</option>" for tid, tname in tier_filters])
    
    def badge(status: str) -> str:
        st = status.lower()
        if st == "yes":
            return "<span style='color: #059669; font-weight: 700;' title='Runs natively'>🟢 Fit</span>"
        elif st == "tight":
            return "<span style='color: #d97706; font-weight: 700;' title='Constrained context / swap risk'>🟡 Tight</span>"
        else:
            return "<span style='color: #dc2626; font-weight: 700;' title='Out of Memory / Exceeds VRAM'>🔴 OOM</span>"
    
    rows_html = []
    for idx, r in df_ram.iterrows():
        bg = "#f8fafc" if idx % 2 == 0 else "#ffffff"
        mkey = r["model_key"]
        mname = r["model_name"]
        arch_badge = f"<span style='background: {('#eff6ff' if r['arch_structure']=='MoE' else '#f8fafc')}; color: {('#1d4ed8' if r['arch_structure']=='MoE' else '#334155')}; padding: 1px 5px; border-radius: 4px; font-weight: 600;'>{r['arch_structure']}</span>"
        attn_badge = f"<span style='background: {('#fef3c7' if r['attention_mech']=='MLA' else '#f1f5f9')}; color: {('#b45309' if r['attention_mech']=='MLA' else '#475569')}; padding: 1px 5px; border-radius: 4px; font-weight: 600;'>{r['attention_mech']}</span>"
        
        # Tier fit flags for JS filter
        f32 = "1" if r["fits_32gb"] in ["yes", "tight"] else "0"
        f64 = "1" if r["fits_64gb"] in ["yes", "tight"] else "0"
        f128 = "1" if r["fits_128gb"] in ["yes", "tight"] else "0"
        f256 = "1" if r["fits_256gb"] in ["yes", "tight"] else "0"
        f512 = "1" if r["fits_512gb"] in ["yes", "tight"] else "0"
        f1tb = "1" if r["fits_1tb"] in ["yes", "tight"] else "0"
        
        row_str = (
            f"<tr style='background: {bg}; border-bottom: 1px solid #f1f5f9;' onmouseover=\"this.style.background='#f1f5f9'\" onmouseout=\"this.style.background='{bg}'\" data-f32='{f32}' data-f64='{f64}' data-f128='{f128}' data-f256='{f256}' data-f512='{f512}' data-f1tb='{f1tb}'>" +
            f"<td style='padding: 8px 10px; font-weight: 600; color: #0f172a;' data-val='{mkey}'>{mname}<br><span style='font-size: 10px; color: #64748b;'>{r['lab']}</span></td>" +
            f"<td style='padding: 8px 8px; text-align: center;' data-val='{r['total_params_b']}'><b>{r['total_params_b']:.1f}B</b><br><span style='font-size: 10px; color: #64748b;'>({r['active_params_b']:.1f}B act)</span></td>" +
            f"<td style='padding: 8px 6px; text-align: center;' data-val='{r['arch_structure']}'>{arch_badge}<br>{attn_badge}</td>" +
            f"<td style='padding: 8px 8px; text-align: center; color: #64748b;' data-val='{r['total_fp16_gb']}'>{r['total_fp16_gb']:.0f} GB</td>" +
            f"<td style='padding: 8px 8px; text-align: center; color: #334155;' data-val='{r['total_fp8_gb']}'>{r['total_fp8_gb']:.0f} GB</td>" +
            f"<td style='padding: 8px 8px; text-align: center; font-weight: 700; color: #0284c7;' data-val='{r['total_q4_gb']}'>{r['total_q4_gb']:.0f} GB</td>" +
            f"<td style='padding: 8px 8px; text-align: center; font-weight: 700; color: #047857;' data-val='{r['min_ram_tier_q4']}'>{r['min_ram_tier_q4']}</td>" +
            f"<td style='padding: 8px 6px; text-align: center;'>{badge(r['fits_32gb'])}</td>" +
            f"<td style='padding: 8px 6px; text-align: center;'>{badge(r['fits_64gb'])}</td>" +
            f"<td style='padding: 8px 6px; text-align: center;'>{badge(r['fits_128gb'])}</td>" +
            f"<td style='padding: 8px 6px; text-align: center;'>{badge(r['fits_256gb'])}</td>" +
            f"<td style='padding: 8px 6px; text-align: center;'>{badge(r['fits_512gb'])}</td>" +
            f"<td style='padding: 8px 6px; text-align: center;'>{badge(r['fits_1tb'])}</td>" +
            f"<td style='padding: 8px 10px; font-size: 11px; color: #475569;'>{r['optimal_target_hardware']}</td>" +
            "</tr>"
        )
        rows_html.append(row_str)
    
    table_body = "\n".join(rows_html)
    
    template = """
<div id='ram-widget-container' style='font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; background: #ffffff; border: 1px solid #e2e8f0; border-radius: 12px; padding: 18px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.05); margin: 20px 0;'>
    <div style='display: flex; justify-content: space-between; align-items: flex-start; flex-wrap: wrap; gap: 12px; margin-bottom: 14px;'>
        <div>
            <h3 style='margin: 0 0 4px 0; font-size: 17px; font-weight: 700; color: #0f172a;'>Open-Weights Hardware Residency & RAM Sizing Matrix</h3>
            <p style='margin: 0; font-size: 12px; color: #64748b;'>Click <b>any column header</b> to sort ascending/descending. Filter by hardware tier memory limits.</p>
        </div>
        <div style='display: flex; gap: 8px; align-items: center; flex-wrap: wrap;'>
            <input id='ram-search-box' type='text' placeholder='🔍 Search model, lab...' style='padding: 6px 10px; font-size: 12px; border: 1px solid #cbd5e1; border-radius: 6px; width: 170px; outline: none;' onkeyup='filterRamTable()'>
            <select id='ram-tier-select' style='padding: 6px 10px; font-size: 12px; border: 1px solid #cbd5e1; border-radius: 6px; background: #ffffff; color: #0f172a; cursor: pointer; outline: none;' onchange='filterRamTable()'>
                __OPT_HTML__
            </select>
        </div>
    </div>
    <div style='overflow-x: auto; max-height: 520px; border: 1px solid #e2e8f0; border-radius: 8px;'>
        <table id='ram-sortable-table' style='width: 100%; border-collapse: collapse; font-size: 11.5px; text-align: left;'>
            <thead style='position: sticky; top: 0; background: #1e293b; color: #f8fafc; z-index: 10;'>
                <tr>
                    <th onclick="sortRamTable(0, 'str')" style='padding: 10px 10px; cursor: pointer; user-select: none; white-space: nowrap;'>Model <span id='ram-icon-0'>↕</span></th>
                    <th onclick="sortRamTable(1, 'num')" style='padding: 10px 8px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>Params <span id='ram-icon-1'>↕</span></th>
                    <th onclick="sortRamTable(2, 'str')" style='padding: 10px 6px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>Arch <span id='ram-icon-2'>↕</span></th>
                    <th onclick="sortRamTable(3, 'num')" style='padding: 10px 8px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>FP16 VRAM <span id='ram-icon-3'>↕</span></th>
                    <th onclick="sortRamTable(4, 'num')" style='padding: 10px 8px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>FP8 VRAM <span id='ram-icon-4'>↕</span></th>
                    <th onclick="sortRamTable(5, 'num')" style='padding: 10px 8px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>Q4 VRAM <span id='ram-icon-5'>↕</span></th>
                    <th onclick="sortRamTable(6, 'str')" style='padding: 10px 8px; cursor: pointer; user-select: none; text-align: center; white-space: nowrap;'>Min Tier <span id='ram-icon-6'>↕</span></th>
                    <th style='padding: 10px 6px; text-align: center; white-space: nowrap;'>32GB</th>
                    <th style='padding: 10px 6px; text-align: center; white-space: nowrap;'>64GB</th>
                    <th style='padding: 10px 6px; text-align: center; white-space: nowrap;'>128GB</th>
                    <th style='padding: 10px 6px; text-align: center; white-space: nowrap;'>256GB</th>
                    <th style='padding: 10px 6px; text-align: center; white-space: nowrap;'>512GB</th>
                    <th style='padding: 10px 6px; text-align: center; white-space: nowrap;'>1TB+</th>
                    <th style='padding: 10px 10px; white-space: nowrap;'>Optimal Target Hardware</th>
                </tr>
            </thead>
            <tbody id='ram-tbody'>
                __TABLE_BODY__
            </tbody>
        </table>
    </div>
    <div style='display: flex; justify-content: space-between; align-items: center; margin-top: 10px; font-size: 11px; color: #64748b;'>
        <div id='ram-status-text'>Showing 16 of 16 open-weights models</div>
        <div>🟢 Fit = Full native headroom | 🟡 Tight = Swap risk / bounded context | 🔴 OOM = Breaches Residency Wall</div>
    </div>
</div>
<script>
var ramCurCol = -1; var ramCurAsc = false;
function sortRamTable(idx, typ) {
    var tb = document.getElementById('ram-tbody'); if (!tb) return;
    var rows = Array.from(tb.querySelectorAll('tr'));
    if (ramCurCol === idx) { ramCurAsc = !ramCurAsc; } else { ramCurCol = idx; ramCurAsc = (typ === 'str'); }
    for (var i = 0; i <= 6; i++) {
        var ic = document.getElementById('ram-icon-' + i);
        if (ic) { ic.innerHTML = (i === idx) ? (ramCurAsc ? '▲' : '▼') : '↕'; ic.style.color = (i === idx) ? '#38bdf8' : '#94a3b8'; }
    }
    rows.sort(function(a, b) {
        var vA = a.children[idx].getAttribute('data-val') || a.children[idx].innerText.trim();
        var vB = b.children[idx].getAttribute('data-val') || b.children[idx].innerText.trim();
        if (typ === 'num') {
            var nA = parseFloat(vA) || 0; var nB = parseFloat(vB) || 0;
            return ramCurAsc ? (nA - nB) : (nB - nA);
        } else {
            return ramCurAsc ? vA.localeCompare(vB) : vB.localeCompare(vA);
        }
    });
    rows.forEach(function(r) { tb.appendChild(r); });
}
function filterRamTable() {
    var q = (document.getElementById('ram-search-box').value || '').toLowerCase();
    var tier = (document.getElementById('ram-tier-select').value || 'all');
    var tb = document.getElementById('ram-tbody'); if (!tb) return;
    var rows = tb.querySelectorAll('tr'); var count = 0;
    rows.forEach(function(r) {
        var txt = r.innerText.toLowerCase();
        var mQ = txt.indexOf(q) !== -1;
        var mTier = true;
        if (tier === '32gb') mTier = (r.getAttribute('data-f32') === '1');
        else if (tier === '64gb') mTier = (r.getAttribute('data-f64') === '1');
        else if (tier === '128gb') mTier = (r.getAttribute('data-f128') === '1');
        else if (tier === '256gb') mTier = (r.getAttribute('data-f256') === '1');
        else if (tier === '512gb') mTier = (r.getAttribute('data-f512') === '1');
        else if (tier === '1tb') mTier = (r.getAttribute('data-f1tb') === '1' && r.getAttribute('data-f512') === '0');
        if (mQ && mTier) { r.style.display = ''; count++; } else { r.style.display = 'none'; }
    });
    var st = document.getElementById('ram-status-text'); if (st) st.innerText = 'Showing ' + count + ' of ' + rows.length + ' open-weights models';
}
</script>
"""
    return template.replace("__OPT_HTML__", opt_html).replace("__TABLE_BODY__", table_body)

def display_sortable_ram_residency_table(data_registry: dict):
    """Renders the interactive open-weights hardware residency table."""
    df_ram = data_registry["open_weights_ram"].copy()
    html_widget = generate_sortable_ram_table_html(df_ram)
    display(HTML(html_widget))

def create_ram_residency_chart(data_registry: dict) -> go.Figure:
    """Generates a Plotly visualization illustrating VRAM residency requirements vs. hardware memory thresholds."""
    df_ram = data_registry["open_weights_ram"].sort_values("total_params_b", ascending=True).copy()
    
    fig = go.Figure()
    
    # 1. FP16 Bar
    fig.add_trace(go.Bar(
        y=df_ram["model_name"],
        x=df_ram["total_fp16_gb"],
        name="Uncompressed FP16 (16-bit)",
        orientation="h",
        marker=dict(color="#94a3b8", line=dict(color="#475569", width=1)),
        hovertemplate="<b>%{y}</b><br>FP16 Total VRAM: %{x:.0f} GB<extra></extra>"
    ))
    
    # 2. FP8 Bar
    fig.add_trace(go.Bar(
        y=df_ram["model_name"],
        x=df_ram["total_fp8_gb"],
        name="Near-Lossless FP8 (8-bit)",
        orientation="h",
        marker=dict(color="#3b82f6", line=dict(color="#1d4ed8", width=1)),
        hovertemplate="<b>%{y}</b><br>FP8 Total VRAM: %{x:.0f} GB<extra></extra>"
    ))
    
    # 3. Q4 Bar
    fig.add_trace(go.Bar(
        y=df_ram["model_name"],
        x=df_ram["total_q4_gb"],
        name="Quantized Q4_K_M / NVFP4 (4-bit)",
        orientation="h",
        marker=dict(color="#10b981", line=dict(color="#047857", width=1)),
        hovertemplate="<b>%{y}</b><br>Q4 Total VRAM: %{x:.0f} GB<extra></extra>"
    ))
    
    # Hardware tier vertical threshold lines
    thresholds = [
        (32, "32 GB (Laptop / Single 24GB GPU)", "#64748b"),
        (64, "64 GB (Workstation / Dual GPUs)", "#f59e0b"),
        (128, "128 GB (DGX Spark / Studio Max)", "#10b981"),
        (256, "256 GB (Mac Studio Ultra 256GB)", "#0284c7"),
        (512, "512 GB (Mac Studio Ultra 512GB)", "#8b5cf6"),
        (1024, "1 TB+ (Datacenter 8-GPU Clusters)", "#ef4444")
    ]
    
    shapes = []
    annotations = []
    for x_val, label, color in thresholds:
        shapes.append(dict(
            type="line",
            x0=x_val, x1=x_val, y0=-0.5, y1=len(df_ram)-0.5,
            line=dict(color=color, width=1.8, dash="dot")
        ))
        annotations.append(dict(
            x=np.log10(x_val),
            y=len(df_ram)-0.2,
            xref="x", yref="y",
            text=f"<b>{x_val}GB</b>",
            showarrow=False,
            font=dict(size=10, color=color),
            bgcolor="rgba(255, 255, 255, 0.85)",
            bordercolor=color, borderwidth=1
        ))
    
    fig.update_layout(
        barmode="group",
        title="<b>Open-Weights VRAM Residency Wall vs. Hardware Memory Tiers</b><br><sup>Total System RAM / VRAM Required Across Quantizations (FP16, FP8, Q4) with 32k KV Cache</sup>",
        xaxis=dict(
            type="log",
            title="Total Memory Required (GB, Log Scale)",
            tickvals=[16, 32, 64, 128, 256, 512, 1024, 1500],
            ticktext=["16GB", "32GB", "64GB", "128GB", "256GB", "512GB", "1TB", "1.5TB"],
            gridcolor="#f1f5f9",
            range=[np.log10(12), np.log10(1600)]
        ),
        yaxis=dict(title="Model Identifier (Ascending Scale)", gridcolor="#f1f5f9"),
        shapes=shapes,
        template="plotly_white",
        height=720,
        margin=dict(l=140, r=20, t=80, b=50),
        legend=dict(orientation="h", y=1.05, x=0.5, xanchor="center", bgcolor="rgba(255,255,255,0.85)")
    )
    return fig

# 1. Render Interactive Sortable Table
print("Rendering Section 7: Open-Weights Hardware Residency Table (Click Header to Sort)...\n")
display_sortable_ram_residency_table(datasets)

# 2. Render Residency Wall Chart
print("\nRendering Section 7: Open-Weights VRAM Residency Wall Chart...")
fig_ram = create_ram_residency_chart(datasets)
fig_ram.show()


### 8. Interactive Data Science Scratchpad
Slice, model, or compute custom metrics across the unified model register and hardware residency matrix below:

In [ ]:
def query_model_catalogue(
    df_cat: pd.DataFrame,
    lab: str = None,
    open_weight: bool = None,
    arch: str = None,
    attn: str = None,
    min_swe_bench: float = None
) -> pd.DataFrame:
    """Programmatically slices and filters the unified model catalogue."""
    res = df_cat.copy()
    if lab is not None:
        res = res[res["lab"].str.contains(lab, case=False, na=False)]
    if open_weight is not None:
        res = res[res["open_weight"] == open_weight]
    if arch is not None:
        res = res[res["arch"].str.upper() == arch.upper()]
    if attn is not None:
        res = res[res["attn"].str.upper() == attn.upper()]
    if min_swe_bench is not None:
        res = res[res["swe_bench"] >= min_swe_bench]
    return res[["model_key", "lab", "weights_label", "release_date", "params", "arch", "attn", "swe_bench_str", "terminal_bench_str", "aa_index_str", "price_str"]]

def query_ram_requirements(
    df_ram: pd.DataFrame,
    max_ram_tier_gb: int = None,
    quant_level: str = "Q4",
    arch: str = None
) -> pd.DataFrame:
    """Queries models that fit within a specified hardware RAM capacity."""
    res = df_ram.copy()
    col_map = {"FP16": "total_fp16_gb", "FP8": "total_fp8_gb", "Q5": "total_q5_gb", "Q4": "total_q4_gb", "Q3": "total_q3_gb"}
    q_col = col_map.get(quant_level.upper(), "total_q4_gb")
    if max_ram_tier_gb is not None:
        res = res[res[q_col] <= max_ram_tier_gb]
    if arch is not None:
        res = res[res["arch_structure"].str.upper() == arch.upper()]
    return res[["model_name", "lab", "total_params_b", "active_params_b", "arch_structure", "attention_mech", q_col, "min_ram_tier_q4", "optimal_target_hardware"]].sort_values(q_col, ascending=False)

# Demonstration Queries
df_ram_master = datasets["open_weights_ram"]

print("=== Models that Fit on 256 GB Workstations in Q4 (Mac Studio Ultra 256GB / Dual Spark) ===")
print(query_ram_requirements(df_ram_master, max_ram_tier_gb=256, quant_level="Q4")[["model_name", "total_params_b", "total_q4_gb", "optimal_target_hardware"]])

print("\n=== Models that Fit on 128 GB Workstations in Q4 (DGX Spark / Studio Max 128GB) ===")
print(query_ram_requirements(df_ram_master, max_ram_tier_gb=128, quant_level="Q4")[["model_name", "total_params_b", "total_q4_gb", "optimal_target_hardware"]])

print("\n=== Models that Fit on 32 GB Laptops (MacBook Pro / Single RTX 4090) in Q4 ===")
print(query_ram_requirements(df_ram_master, max_ram_tier_gb=32, quant_level="Q4")[["model_name", "total_params_b", "total_q4_gb", "optimal_target_hardware"]])
